# Credit Card Fraud Detection
## Detection Transaction Fraud and Comparing Algorithms Trained On Real and Synthetic Data

## Introduction

The goal of this project is to scope, prep, analyze real-world and synthetic data, train machine learning models to classify transactions as fraudulent or not and compare perfomance of models trained on real-world credit card data and synthetically generated credit card data

## Scope

### Project Goals

The main research topic of this project is to use machinle learning techniques to classifiy credit card transactions as fraudulent or not and compare models trained on real-world and synthetic data. This is important because many credit card companies use machine learning algorithms to flag transactions as fraudulent or not so that customers are not charged for items they did not purchase



### Data

The project uses two datasets. The first dataset was privided by `Kaggle.com` and contains over 200k real-word transactions made by European cardholders in 2013. The dataset presents transactions that occurred in two days, where we have 492 frauds out of 284,807 transactions. The dataset is highly unbalanced, the positive class (frauds) account for 0.172% of all transactions. It contains only numerical input variables which are the result of a PCA transformation. Unfortunately, due to confidentiality issues, no additional information about the original features was found. The only features that haven't been tansformed are time, and transaction amount.

Dataset Source: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud

The second dataset contains 10000 synthetically generated transactions with patterns imitating real-word credit card fraud where the positive class acounts for only 2% of the data (8000 legit, 2000 fraud). The generator script is included with the project. The dataset has been split into `train.csv` with 80% of the data for training models, and `test.csv` with 20% for testing the models.

### Evaluation

The project will conclude with the evaluation of the machine learning model selected with a validation data set. The output of the predictions can be checked through a confusion matrix, and metrics such as accuracy, precision, recall, F1 and Kappa scores.

## Import Python Modules

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Load Datasets

In [2]:
transaction_data_synthetic_train = pd.read_csv("data/train.csv")
transaction_data_real = pd.read_csv("data/credit_card_data_real.zip", compression="zip")

In [3]:
print(transaction_data_synthetic_train.head())

       amount  hour  day_of_week merchant_category  is_fraud
0  214.056124    10            2            retail         0
1  470.044641    14            1           grocery         0
2   59.567221     9            4           grocery         0
3   28.190464    21            0            retail         0
4   45.258306    16            3           grocery         0


In [4]:
print(transaction_data_real.head())

   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 -0.206010   

        V26       V27       V28 

### Data Characteristics

The synthetic transaction dataset contains 5 columns and 10000 rows

The columns of the dataset include:

- **amount** - The amount spend
- **hour** - Time of day the transaction was made (0 - 23)
- **day_of_week** - In which day of the week the transaction was made (0 -> Monday, 6 -> Sunday)
- **merchant_category** - The type of merchat the transaction was made to
- **is_fraud** - The target variable indicating presense (1) or abscense of fraud (0)

The real dataset contains 284,807 transactions and 31 columns.

Most of the columns of the dataset have been transformed with PCA due to privacy concerns and no information was found as to what this columns may represent. The only untouched columns are

- **time** - Number of seconds elapsed between the current transaction and the first transaction in the dataset.
- **amount** - The amount spent in EUR
- **Class** - The target valiable indicating presense(1) or abscense of fraud (0)

## Exploratory Data Analysis

### Shape, Summary Statistics, Data Types

The shape of the training split of the synthetic dataset is as expected. As seen in the summary statistics some features are  on different scales and may need to be scaled and merchant category needs to be encoded before modeling.

In [5]:
print("Shape: ", transaction_data_synthetic_train.shape)
print("Summary Statistics: \n", transaction_data_synthetic_train.describe())
print("Data Types: \n", transaction_data_synthetic_train.dtypes)

Shape:  (8000, 5)
Summary Statistics: 
              amount         hour  day_of_week     is_fraud
count   8000.000000  8000.000000  8000.000000  8000.000000
mean      86.470016    11.340000     3.048625     0.020000
std      315.417790     6.973125     1.980089     0.140009
min        0.299129     0.000000     0.000000     0.000000
25%       15.294651     5.000000     1.000000     0.000000
50%       33.924469    11.000000     3.000000     0.000000
75%       76.880313    17.000000     5.000000     0.000000
max    13446.047675    23.000000     6.000000     1.000000
Data Types: 
 amount               float64
hour                   int64
day_of_week            int64
merchant_category        str
is_fraud               int64
dtype: object


The shape of the real dataset is also as expected. As per the summary statistics we ca see that here some features should also be scaled before modeling. 

In [6]:
print("Shape: ", transaction_data_real.shape)
print("Summary Statistics: \n", transaction_data_real.describe())
print("Data Types: \n", transaction_data_real.dtypes)

Shape:  (284807, 31)
Summary Statistics: 
                 Time            V1            V2            V3            V4  \
count  284807.000000  2.848070e+05  2.848070e+05  2.848070e+05  2.848070e+05   
mean    94813.859575  1.175161e-15  3.384974e-16 -1.379537e-15  2.094852e-15   
std     47488.145955  1.958696e+00  1.651309e+00  1.516255e+00  1.415869e+00   
min         0.000000 -5.640751e+01 -7.271573e+01 -4.832559e+01 -5.683171e+00   
25%     54201.500000 -9.203734e-01 -5.985499e-01 -8.903648e-01 -8.486401e-01   
50%     84692.000000  1.810880e-02  6.548556e-02  1.798463e-01 -1.984653e-02   
75%    139320.500000  1.315642e+00  8.037239e-01  1.027196e+00  7.433413e-01   
max    172792.000000  2.454930e+00  2.205773e+01  9.382558e+00  1.687534e+01   

                 V5            V6            V7            V8            V9  \
count  2.848070e+05  2.848070e+05  2.848070e+05  2.848070e+05  2.848070e+05   
mean   1.021879e-15  1.494498e-15 -5.620335e-16  1.149614e-16 -2.414189e-15   

### Duplicates and Missing Values

The synthetic dataset doesn't contain any missing values, nor duplicates.

In [7]:
print(transaction_data_synthetic_train.isna().sum())
print(transaction_data_synthetic_train.duplicated().sum())

amount               0
hour                 0
day_of_week          0
merchant_category    0
is_fraud             0
dtype: int64
0


The real dataset doesn't have any missing values but it has 1081 dupclicated transactions which have to be removed

In [10]:
print(transaction_data_real.isna().sum())
print("Duplicates: ", transaction_data_real.duplicated().sum())

Time      0
V1        0
V2        0
V3        0
V4        0
V5        0
V6        0
V7        0
V8        0
V9        0
V10       0
V11       0
V12       0
V13       0
V14       0
V15       0
V16       0
V17       0
V18       0
V19       0
V20       0
V21       0
V22       0
V23       0
V24       0
V25       0
V26       0
V27       0
V28       0
Amount    0
Class     0
dtype: int64
Duplicates:  1081


After dropping the duplicated rows we are left with 283726 unique records.

In [12]:
transaction_data_real = transaction_data_real.drop_duplicates()
print(transaction_data_real.shape)

(283726, 31)


### Target Variable Distributions

Despite the uneven number of obesrvetions in the positive class of the target variable in the train and test set, we can conclude that the generation of the synthetic dataset was successfull

In [14]:
transaction_data_synthetic_test = pd.read_csv("data/test.csv")

print(transaction_data_synthetic_train.is_fraud.value_counts())
print(transaction_data_synthetic_test.is_fraud.value_counts())

is_fraud
0    7840
1     160
Name: count, dtype: int64
is_fraud
0    1960
1      40
Name: count, dtype: int64


Some of the obesrvetions form the possitive class in the real dataset have been lost when droping duplicate rows. We are left with only 473 fraud transactions.

In [16]:
print(transaction_data_real["Class"].value_counts())

Class
0    283253
1       473
Name: count, dtype: int64
